# Explore ML-ready distributions, correlation, and embeddings

**Goal.** Explore a bounded, deterministic payment world through computed tables and plots.

**Audience.** Data scientists and ML engineers learning FraudTwin's first milestones.

**Prerequisites.** Python 3.12+, FraudTwin, and Polars. Optional plotting cells can install notebook packages.

**Source size.** 1,000 logical payments; outputs are reproducible and temporary.

**Offline path.** Analysis runs without Kafka, databases, or cloud services.


## Optional notebook packages
Run this only for richer plots or t-SNE output.

```bash
!pip install matplotlib scikit-learn
```


## 1. Generate the source world


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import polars as pl

import fraudtwin
from fraudtwin.config import SimulationRunConfig, load_config

root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "minimal.yaml").is_file()
)
base = load_config(root / "configs" / "minimal.yaml")
values = base.model_dump(mode="python")
values["payments"]["daily_target"] = 100
values["simulation"]["duration_days"] = 10
values["population"].update(
    {
        "customers": 100,
        "accounts": 200,
        "cards": 200,
        "merchants": 30,
        "devices": 200,
        "pix_keys": 100,
    }
)
values["behavior"]["amount_max"] = 100
values["simulation"]["seed"] = 2501
config = SimulationRunConfig.model_validate(values)
data = fraudtwin.generate(config)
print({"run_id": data.run_id, "payments": len(data.behavior.payments)})
assert len(data.behavior.payments) >= 1_000

## 2. Analysis step


In [ ]:
frame = data.require_dataset().frame
print({"rows": frame.height, "columns": frame.width})
assert frame.height > 0

## 3. Analysis step


In [ ]:
numeric = [
    name
    for name, dtype in frame.schema.items()
    if dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
]
print({"numeric_features": numeric[:12]})

## 4. Analysis step


In [ ]:
missingness = frame.select([pl.col(name).is_null().mean().alias(name) for name in numeric[:8]])
missingness

## 5. Analysis step


In [ ]:
label_balance = (
    frame.group_by("label").len().sort("len", descending=True)
    if "label" in frame.columns
    else pl.DataFrame()
)
print(label_balance)

## 6. Analysis step


In [ ]:
feature_summary = frame.select([pl.col(name).mean().alias(f"{name}_mean") for name in numeric[:6]])
feature_summary

## 7. Analysis step


In [ ]:
correlation = frame.select(numeric[:6]).corr() if len(numeric) >= 2 else pl.DataFrame()
print(correlation)

## 8. Analysis step


In [ ]:
time_splits = (
    frame.group_by("split").len().sort("split") if "split" in frame.columns else pl.DataFrame()
)
print(time_splits)

## 9. Analysis step


In [ ]:
embedding_input = frame.select(numeric[:2]).drop_nulls() if len(numeric) >= 2 else pl.DataFrame()
print(
    {
        "embedding_rows": embedding_input.height,
        "method": "optional sklearn t-SNE; deterministic tabular fallback",
    }
)

## 10. Analysis step


In [ ]:
with TemporaryDirectory(prefix="fraudtwin-viz-27-") as tmp:
    report = Path(tmp) / "feature-summary.json"
    report.write_text(
        json.dumps({"numeric": numeric, "rows": frame.height}, default=str), encoding="utf-8"
    )
    print({"artifact": str(report), "bytes": report.stat().st_size})
    assert report.stat().st_size < 20_000